# 02 – Topic Modelling (BTM)
Train Biterm Topic Models per party and user group.

**Inputs** : `outputs/interactions/*.json`  
**Outputs**: `outputs/topical_models/<group>/<party>/`

In [ ]:
import logging, sys, os
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from src.data.loaders import load_config
from src.topical_models.btm_model import (
    preprocess, build_phrase_models, apply_bigrams,
    lemmatize, fit_btm, save_btm_results,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
cfg = load_config('configs/config.yaml')
print('Config loaded.')

## Pipeline
For each `(group, party)` pair:
1. Load interactions JSON
2. Filter to non-RT tweets
3. Preprocess and tokenise
4. Build bigram phrases
5. Lemmatise
6. Fit BTM (coherence-selected topic count)
7. Save results

In [ ]:
groups = [
    ('celebs',      cfg['outputs']['interactions_dir'] + 'celeb_interactions.json'),
    ('politicians', cfg['outputs']['interactions_dir'] + 'pol_interactions.json'),
]
parties = ['INC', 'BJP']
btm_cfg = cfg['btm']
phr_cfg = cfg['phrases']
topic_range = range(btm_cfg['min_topic_range'], btm_cfg['max_topic_range'], btm_cfg['topic_step'])

for group, src_path in groups:
    print(f'\n=== {group} ===')
    interactions = pd.read_json(src_path)

    for party in parties:
        print(f'  --- {party} ---')
        out_dir = os.path.join(cfg['outputs']['topic_models_dir'], group, party)

        # Skip if already computed
        if os.path.exists(os.path.join(out_dir, 'rt_coherence.csv')):
            print(f'  Already computed, skipping.')
            continue

        subset = interactions[
            (interactions['type'] != 'rt') & (interactions['party'] == party)
        ].copy()
        subset['tokenized_text'] = subset['full_text'].apply(preprocess)
        subset = subset[subset['tokenized_text'].apply(len) > 0]
        if subset.empty:
            print(f'  No data for {party}, skipping.')
            continue

        data_words = list(subset['tokenized_text'])
        bigram_mod, _ = build_phrase_models(
            data_words,
            min_count=phr_cfg['min_count'],
            bigram_threshold=phr_cfg['bigram_threshold'],
        )
        data_bigrams  = apply_bigrams(data_words, bigram_mod)
        data_lemmas   = lemmatize(data_bigrams)
        subset['words'] = [' '.join(t) for t in data_lemmas]

        topics, summary, _ = fit_btm(subset, topic_range,
                                      iterations=btm_cfg['iterations'],
                                      max_df=btm_cfg['max_df'],
                                      min_df=btm_cfg['min_df'])
        coherence_scores = list(summary['coherence'])
        save_btm_results(subset, topics, summary, coherence_scores, out_dir)

print('\nAll topic models trained.')

## Inspect a model

In [ ]:
import pandas as pd
sample_path = os.path.join(cfg['outputs']['topic_models_dir'], 'politicians', 'INC', 'rt_overall_topics.csv')
if os.path.exists(sample_path):
    pd.read_csv(sample_path)